# First filtering

In [2]:
!pip install ijson
import os, ijson, json, time, glob
from typing import Any, Iterator, Dict, Set


In [3]:
INPUT_FILE = "./research_products_data/research_products_complete.json"
OUT_IDS = "./research_products_data/unique_ids_research_products_orgs.txt"

def _is_ndjson(path: str) -> bool:
    if path.lower().endswith(".ndjson"):
        return True
    with open(path, "r", encoding="utf-8") as f:
        while True:
            c = f.read(1)
            if not c:
                return False
            if not c.isspace():
                return c != "["  # se inizia con [, è JSON array (non NDJSON)

def iter_products_streaming(path: str) -> Iterator[Dict]:
    if _is_ndjson(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    yield json.loads(line)
                except Exception:
                    continue
    else:
        try:
            import ijson  # pip install ijson
        except ImportError as e:
            raise RuntimeError("Installa ijson: pip install ijson") from e
        with open(path, "r", encoding="utf-8") as f:
            for obj in ijson.items(f, "item"):
                yield obj

assert os.path.exists(INPUT_FILE), f"File non trovato: {INPUT_FILE}"

start = time.time()
uniq_ids: Set[str] = set()
total = 0

for obj in iter_products_streaming(INPUT_FILE):
    total += 1
    _id = obj.get("id")
    if _id:
        uniq_ids.add(str(_id))

elapsed = time.time() - start
print(f"Scansionati {total} record in {elapsed:.1f}s. ID unici: {len(uniq_ids)}")

# salva su file (uno per riga)
os.makedirs(os.path.dirname(OUT_IDS), exist_ok=True)
with open(OUT_IDS, "w", encoding="utf-8") as f:
    for _id in sorted(uniq_ids):
        f.write(f"{_id}\n")

# anteprima
print("Esempi:", list(sorted(uniq_ids))[:5])
print("Salvati in:", OUT_IDS)



Scansionati 265798 record in 34.5s. ID unici: 265798
Esempi: ['57a035e5b1ae::7f8397cd75dc38ea1db83562a7118089', '57a035e5b1ae::965dd287678046e6b834e3d0a40f7c21', 'RECOLECTA___::bb219f8c63c3d7ab6134cdb87841b354', 'arXiv_dedup_::0883cb0f298698315282e0b6af68e825', 'arXiv_dedup_::3c883b1ed1983e978d5c720463c54c2c']
Salvati in: ./research_products_data/unique_ids_research_products_orgs.txt


In [4]:
def load_unique_ids(path: str) -> Set[str]:
    assert os.path.exists(path), f"File non trovato: {path}"
    with open(path, "r", encoding="utf-8") as f:
        return {line.strip() for line in f if line.strip()}

def _is_ndjson(path: str) -> bool:
    if path.lower().endswith(".ndjson"):
        return True
    with open(path, "r", encoding="utf-8") as f:
        while True:
            c = f.read(1)
            if not c:
                return False
            if not c.isspace():
                return c != "["

def iter_json_objects(path: str) -> Iterator[Dict[str, Any]]:
    if _is_ndjson(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    if isinstance(obj, dict):
                        yield obj
                except Exception:
                    continue
    else:
        try:
            import ijson
        except ImportError as e:
            raise RuntimeError("Installa ijson: pip install ijson") from e
        with open(path, "r", encoding="utf-8") as f:
            for obj in ijson.items(f, "item"):
                if isinstance(obj, dict):
                    yield obj

def get_id_value(v: Any) -> str:
    if isinstance(v, str):
        return v.strip()
    if isinstance(v, dict):
        for k in ("id", "ID"):
            if k in v and isinstance(v[k], str):
                return v[k].strip()
    return ""

def append_ndjson(path: str, obj: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def filter_relations_by_ids(unique_ids_file: str, rel_dir: str, out_file: str):
    ids = load_unique_ids(unique_ids_file)
    print(f"Loaded unique IDs: {len(ids)}")

    paths = sorted(glob.glob(os.path.join(rel_dir, "**", "*.json"), recursive=True))
    print(f"JSON files found: {len(paths)} in {rel_dir}")

    t0 = time.time()
    kept = 0
    scanned = 0

    if os.path.exists(out_file):
        os.remove(out_file)

    for p in paths:
        for rel in iter_json_objects(p):
            scanned += 1
            s_id = get_id_value(rel.get("source"))
            t_id = get_id_value(rel.get("target"))
            if (s_id and s_id in ids) or (t_id and t_id in ids):
                rel_out = dict(rel)
                rel_out["_source_file"] = os.path.relpath(p)
                append_ndjson(out_file, rel_out)
                kept += 1

    dt = time.time() - t0
    print(f"Scansionate relazioni: {scanned} | mantenute: {kept} | tempo: {dt:.1f}s")
    print(f"Output: {out_file}")

In [12]:

UNIQUE_IDS_FILE = "./research_products_data/unique_ids_research_products_orgs.txt"
REL_DIR = "../relation_data/01/relation"
OUT_FILE = "../relation_data/01/filtered_relations.ndjson"

filter_relations_by_ids(UNIQUE_IDS_FILE, REL_DIR, OUT_FILE)

Loaded unique IDs: 265798
JSON files found: 352 in ../relation_data/01/relation
Scansionate relazioni: 714991879 | mantenute: 1102341 | tempo: 1450.6s
Output: ../relation_data/01/filtered_relations.ndjson


In [15]:
UNIQUE_IDS_FILE = "./research_products_data/unique_ids_research_products_orgs.txt"
REL_DIR = "../relation_data/02/relation"
OUT_FILE = "../relation_data/02/filtered_relations.ndjson"

filter_relations_by_ids(UNIQUE_IDS_FILE, REL_DIR, OUT_FILE)

Loaded unique IDs: 265798
JSON files found: 305 in ../relation_data/02/relation
Scansionate relazioni: 710541571 | mantenute: 1170128 | tempo: 1587.4s
Output: ../relation_data/02/filtered_relations.ndjson


In [7]:
import gc
gc.collect()

1050

In [17]:
UNIQUE_IDS_FILE = "./research_products_data/unique_ids_research_products_orgs.txt"
REL_DIR = "../relation_data/03/relation"
OUT_FILE = "../relation_data/03/filtered_relations.ndjson"

filter_relations_by_ids(UNIQUE_IDS_FILE, REL_DIR, OUT_FILE)

Loaded unique IDs: 265798
JSON files found: 300 in ../relation_data/03/relation
Scansionate relazioni: 714826616 | mantenute: 1131267 | tempo: 1591.2s
Output: ../relation_data/03/filtered_relations.ndjson


In [18]:
gc.collect()

0

In [19]:
UNIQUE_IDS_FILE = "./research_products_data/unique_ids_research_products_orgs.txt"
REL_DIR = "../relation_data/04/relation"
OUT_FILE = "../relation_data/04/filtered_relations.ndjson"

filter_relations_by_ids(UNIQUE_IDS_FILE, REL_DIR, OUT_FILE)

Loaded unique IDs: 265798
JSON files found: 305 in ../relation_data/04/relation
Scansionate relazioni: 717551572 | mantenute: 1146480 | tempo: 1601.0s
Output: ../relation_data/04/filtered_relations.ndjson


In [20]:
gc.collect()

0

In [21]:
UNIQUE_IDS_FILE = "./research_products_data/unique_ids_research_products_orgs.txt"
REL_DIR = "../relation_data/05/relation"
OUT_FILE = "../relation_data/05/filtered_relations.ndjson"

filter_relations_by_ids(UNIQUE_IDS_FILE, REL_DIR, OUT_FILE)

Loaded unique IDs: 265798
JSON files found: 271 in ../relation_data/05/relation
Scansionate relazioni: 704620700 | mantenute: 1160430 | tempo: 1663.3s
Output: ../relation_data/05/filtered_relations.ndjson


In [22]:
gc.collect()

0

In [23]:
UNIQUE_IDS_FILE = "./research_products_data/unique_ids_research_products_orgs.txt"
REL_DIR = "../relation_data/06/relation"
OUT_FILE = "../relation_data/06/filtered_relations.ndjson"

filter_relations_by_ids(UNIQUE_IDS_FILE, REL_DIR, OUT_FILE)

Loaded unique IDs: 265798
JSON files found: 203 in ../relation_data/06/relation
Scansionate relazioni: 801580137 | mantenute: 1692689 | tempo: 1936.5s
Output: ../relation_data/06/filtered_relations.ndjson


In [24]:
gc.collect()

0

In [25]:
UNIQUE_IDS_FILE = "./research_products_data/unique_ids_research_products_orgs.txt"
REL_DIR = "../relation_data/07/relation"
OUT_FILE = "../relation_data/07/filtered_relations.ndjson"

filter_relations_by_ids(UNIQUE_IDS_FILE, REL_DIR, OUT_FILE)

Loaded unique IDs: 265798
JSON files found: 137 in ../relation_data/07/relation
Scansionate relazioni: 505653269 | mantenute: 1217090 | tempo: 1241.7s
Output: ../relation_data/07/filtered_relations.ndjson


In [26]:
gc.collect()

0

In [5]:

UNIQUE_IDS_FILE = "./research_products_data/unique_ids_research_products_orgs.txt"
REL_DIR = "../relation_data/08/relation"
OUT_FILE = "../relation_data/08/filtered_relations.ndjson"

filter_relations_by_ids(UNIQUE_IDS_FILE, REL_DIR, OUT_FILE)

Loaded unique IDs: 265798
JSON files found: 139 in ../relation_data/08/relation
Scansionate relazioni: 460552617 | mantenute: 1232942 | tempo: 1098.3s
Output: ../relation_data/08/filtered_relations.ndjson


In [8]:
gc.collect()

0

In [9]:
UNIQUE_IDS_FILE = "./research_products_data/unique_ids_research_products_orgs.txt"
REL_DIR = "../relation_data/09/relation"
OUT_FILE = "../relation_data/09/filtered_relations.ndjson"

filter_relations_by_ids(UNIQUE_IDS_FILE, REL_DIR, OUT_FILE)

Loaded unique IDs: 265798
JSON files found: 142 in ../relation_data/09/relation
Scansionate relazioni: 457542401 | mantenute: 1254217 | tempo: 985.5s
Output: ../relation_data/09/filtered_relations.ndjson


In [ ]:
gc.collect()

In [10]:
UNIQUE_IDS_FILE = "./research_products_data/unique_ids_research_products_orgs.txt"
REL_DIR = "../relation_data/10/relation"
OUT_FILE = "../relation_data/10/filtered_relations.ndjson"

filter_relations_by_ids(UNIQUE_IDS_FILE, REL_DIR, OUT_FILE)

Loaded unique IDs: 265798
JSON files found: 146 in ../relation_data/10/relation
Scansionate relazioni: 456647253 | mantenute: 1290069 | tempo: 1041.1s
Output: ../relation_data/10/filtered_relations.ndjson


In [11]:
gc.collect()

0

In [12]:
UNIQUE_IDS_FILE = "./research_products_data/unique_ids_research_products_orgs.txt"
REL_DIR = "../relation_data/11/relation"
OUT_FILE = "../relation_data/11/filtered_relations.ndjson"

filter_relations_by_ids(UNIQUE_IDS_FILE, REL_DIR, OUT_FILE)

Loaded unique IDs: 265798
JSON files found: 116 in ../relation_data/11/relation
Scansionate relazioni: 358024805 | mantenute: 1030225 | tempo: 733.4s
Output: ../relation_data/11/filtered_relations.ndjson


In [13]:
gc.collect()

0

In [14]:
UNIQUE_IDS_FILE = "./research_products_data/unique_ids_research_products_orgs.txt"
REL_DIR = "../relation_data/12/relation"
OUT_FILE = "../relation_data/12/filtered_relations.ndjson"

filter_relations_by_ids(UNIQUE_IDS_FILE, REL_DIR, OUT_FILE)

Loaded unique IDs: 265798
JSON files found: 155 in ../relation_data/12/relation
Scansionate relazioni: 455593908 | mantenute: 1351856 | tempo: 806.3s
Output: ../relation_data/12/filtered_relations.ndjson


In [15]:
gc.collect()

0

In [16]:
UNIQUE_IDS_FILE = "./research_products_data/unique_ids_research_products_orgs.txt"
REL_DIR = "../relation_data/13/relation"
OUT_FILE = "../relation_data/13/filtered_relations.ndjson"

filter_relations_by_ids(UNIQUE_IDS_FILE, REL_DIR, OUT_FILE)

Loaded unique IDs: 265798
JSON files found: 64 in ../relation_data/13/relation
Scansionate relazioni: 207001339 | mantenute: 786761 | tempo: 391.1s
Output: ../relation_data/13/filtered_relations.ndjson
